# tinyserve — GPU benchmarks (step 4.6)

Produces the publishable numbers: serial vs static vs continuous batching on a GPU, for GPT-2 and Qwen2.5-0.5B, plus the Hugging Face `generate()` floor and the vLLM ceiling, all on the **same workload and load generator**.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

**Keep this tab open** while it runs (about 2–3 hours in total). Free Colab can disconnect idle sessions; every benchmark saves its results after each run, and cell 8 can download whatever has finished at any time.

## 1. GPU and environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
assert tuple(int(x) for x in torch.__version__.split(".")[:2]) >= (2, 5), "need torch >= 2.5 (SDPA enable_gqa)"

## 2. Get the code

The package lives at the repo root, so it runs from the checkout without `pip install -e .` (which would also try to replace Colab's CUDA build of torch). Only the small dependencies are installed.

In [ ]:
!git clone -q https://github.com/Sathyan26/tinyserve.git
%cd tinyserve
!git log --oneline -1
!pip install -q tokenizers safetensors huggingface_hub fastapi "uvicorn[standard]" prometheus-client transformers matplotlib httpx pyarrow openai

## 3. Correctness on the GPU first

Re-proves on CUDA what the test suite proves on CPU: logits match Hugging Face, and padded batches and the scheduler give the same tokens as running each request alone (fp32 exactly; fp16 exactly or a proven near-tie). **If this cell fails, stop: benchmark numbers from a wrong engine are meaningless.**

In [ ]:
!python -m tinyserve.bench.gpu_check --device cuda

## 4. Headline benchmark: GPT-2 (fp16, 16 slots)  ·  ~45 min

In [ ]:
!python -u -m tinyserve.bench.run_all --device cuda --model gpt2 --dtype float16 --only serving --slots 16

## 5. Headline benchmark: Qwen2.5-0.5B-Instruct (fp16, 16 slots)  ·  ~60 min

In [ ]:
!python -u -m tinyserve.bench.run_all --device cuda --model qwen2.5-0.5b --dtype float16 --only serving --slots 16

## 6. Floor: Hugging Face `generate()`, one request at a time  ·  ~10 min

In [ ]:
!python -u -m tinyserve.bench.hf_floor --device cuda --model gpt2 --dtype float16
!python -u -m tinyserve.bench.hf_floor --device cuda --model qwen2.5-0.5b --dtype float16

## 7. Ceiling: vLLM (optional)  ·  ~30 min

Installs vLLM, serves each model with the same slot count (`--max-num-seqs 16`), and replays the same workload with the same load generator. GPT-2 has no chat template, so vLLM gets `templates/gpt2_chat.jinja`, which a test proves is byte-identical to tinyserve's prompt format.

If vLLM fails to install or start on this GPU, skip this cell: the README then explains the gap to vLLM in words, which the spec allows.

In [ ]:
import subprocess, time, httpx

!pip install -q vllm

def wait_ready(url, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if httpx.get(url + "/v1/models", timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(5)
    return False

for model, served, extra in [
    ("gpt2", "openai-community/gpt2", ["--chat-template", "templates/gpt2_chat.jinja"]),
    ("qwen2.5-0.5b", "Qwen/Qwen2.5-0.5B-Instruct", []),
]:
    log = open(f"vllm_{model}.log", "w")
    proc = subprocess.Popen(
        ["vllm", "serve", served, "--port", "8001", "--dtype", "float16",
         "--max-num-seqs", "16", "--max-model-len", "1024", *extra],
        stdout=log, stderr=subprocess.STDOUT)
    try:
        if not wait_ready("http://127.0.0.1:8001"):
            print(f"vLLM did not start for {model}; see vllm_{model}.log")
            continue
        !python -u -m tinyserve.bench.ceiling --url http://127.0.0.1:8001 --served-model {served} --model {model} --device cuda
    finally:
        proc.terminate(); proc.wait(60)

## 8. Download the results

Zips `docs/results/` and `docs/img/`. Unzip it over your local `E:	inyserve` (it only contains those two folders), check the graphs, then commit.

In [ ]:
!zip -qr gpu_results.zip docs/results docs/img
from google.colab import files
files.download("gpu_results.zip")